In [1]:
import sys
sys.path.insert(0, '/app')

from connectors.clickhouse_client import ClickHouseClient
from config.settings import clickhouse_config

In [2]:
client = ClickHouseClient()


In [3]:
client.execute_query("CREATE DATABASE IF NOT EXISTS bronze")


In [4]:
bronze_snapshot_ddl = """
CREATE TABLE IF NOT EXISTS bronze.snapshot_raw
(
    ref_date Date,
    table_name String,
    primary_key String,
    row_hash String,
    data String,
    ingestion_timestamp DateTime DEFAULT now()
)
ENGINE = MergeTree()
PARTITION BY toYYYYMM(ref_date)
ORDER BY (ref_date, table_name, primary_key)
SETTINGS index_granularity = 8192
"""

client.execute_query(bronze_snapshot_ddl)


In [5]:
bronze_control_ddl = """
CREATE TABLE IF NOT EXISTS bronze.ingestion_control
(
    ref_date Date,
    table_name String,
    schema_name String,
    row_count UInt64,
    ingestion_start DateTime,
    ingestion_end DateTime,
    status String,
    error_message String
)
ENGINE = MergeTree()
PARTITION BY toYYYYMM(ref_date)
ORDER BY (ref_date, table_name)
SETTINGS index_granularity = 8192
"""

client.execute_query(bronze_control_ddl)


In [6]:
client.execute_query("CREATE DATABASE IF NOT EXISTS silver")


In [7]:
silver_delta_ddl = """
CREATE TABLE IF NOT EXISTS silver.delta_events
(
    ref_date Date,
    table_name String,
    primary_key String,
    operation_type Enum8('INSERT' = 1, 'UPDATE' = 2, 'DELETE' = 3),
    row_hash_before String,
    row_hash_after String,
    data_before String,
    data_after String,
    detected_at DateTime DEFAULT now()
)
ENGINE = MergeTree()
PARTITION BY toYYYYMM(ref_date)
ORDER BY (ref_date, table_name, primary_key, operation_type)
SETTINGS index_granularity = 8192
"""

client.execute_query(silver_delta_ddl)


In [8]:
silver_state_ddl = """
CREATE TABLE IF NOT EXISTS silver.current_state
(
    table_name String,
    primary_key String,
    row_hash String,
    data String,
    first_seen_date Date,
    last_seen_date Date,
    is_active UInt8,
    updated_at DateTime DEFAULT now()
)
ENGINE = ReplacingMergeTree(updated_at)
PARTITION BY table_name
ORDER BY (table_name, primary_key)
SETTINGS index_granularity = 8192
"""

client.execute_query(silver_state_ddl)


In [9]:
client.execute_query("CREATE DATABASE IF NOT EXISTS gold")


In [10]:
gold_metrics_ddl = """
CREATE TABLE IF NOT EXISTS gold.daily_change_metrics
(
    ref_date Date,
    table_name String,
    total_inserts UInt64,
    total_updates UInt64,
    total_deletes UInt64,
    total_active_records UInt64,
    calculated_at DateTime DEFAULT now()
)
ENGINE = SummingMergeTree()
PARTITION BY toYYYYMM(ref_date)
ORDER BY (ref_date, table_name)
SETTINGS index_granularity = 8192
"""

client.execute_query(gold_metrics_ddl)


In [11]:
gold_quality_ddl = """
CREATE TABLE IF NOT EXISTS gold.data_quality_metrics
(
    ref_date Date,
    table_name String,
    null_count UInt64,
    duplicate_count UInt64,
    total_records UInt64,
    quality_score Float64,
    calculated_at DateTime DEFAULT now()
)
ENGINE = ReplacingMergeTree(calculated_at)
PARTITION BY toYYYYMM(ref_date)
ORDER BY (ref_date, table_name)
SETTINGS index_granularity = 8192
"""

client.execute_query(gold_quality_ddl)


In [12]:
result = client.execute_query_with_result("SHOW DATABASES")
result.result_rows


[('INFORMATION_SCHEMA',),
 ('bronze',),
 ('default',),
 ('ginf',),
 ('gold',),
 ('information_schema',),
 ('scot',),
 ('siga',),
 ('silver',),
 ('stage',),
 ('system',),
 ('tracker_gold',),
 ('tracker_raw',),
 ('tracker_trusted',),
 ('uk',)]

In [ ]:
client.close()
